# Instella Reasoning Atlas — Colab Runner

**Reasoning or Remembering?** When AMD Instella answers a reasoning problem correctly, is it *reasoning* or *remembering*? This notebook clones the repo and runs the full analysis pipeline: contamination search -> consistency-based Reliability -> training-data attribution -> the Reasoning Reliability Atlas.

It works in two modes:

1. **CPU smoke (no GPU, no downloads)** — runs the entire analysis on bundled example data using a dependency-free embedder. Proves the pipeline end-to-end in ~1 minute. Great for a first run or CI.
2. **Full T4 run** — installs the GPU extras, downloads real benchmarks + Instella training shards from HuggingFace, generates with Instella-3B (4-bit), and builds the real Atlas.

In Colab: **Runtime -> Change runtime type -> T4 GPU** for mode 2 (mode 1 needs no GPU).

## 0. Clone and install

Clone the repository and install the package. The base install is pure-Python and CPU-only.

In [ ]:
import os, subprocess, sys

REPO_URL = os.environ.get('INSTELLA_REPO', 'https://github.com/GIND123/Instella-Reasoning.git')
REPO_DIR = 'Instella-Reasoning'

# Clone only if we are not already inside the repo (e.g. running locally).
if not os.path.exists('pyproject.toml'):
    if not os.path.exists(REPO_DIR):
        subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
print('Working directory:', os.getcwd())

In [ ]:
# Base install (CPU, dependency-free core). Add extras below for the full run.
!pip install -q -e ".[dev]"
print('\nInstalled. CLI check:')
!instella-reasoning --help | head -5

## 1. CPU smoke — run ALL analyses with no GPU and no downloads

This runs the complete orchestrator on the bundled `examples/` data with the hashing embedder + brute-force index. Every stage executes: contamination scan, scoring, accuracy gap, embedding attribution, the Reliability Atlas, and the Markdown report.

In [ ]:
!instella-reasoning run-all --config configs/pipeline/full.yaml

In [ ]:
from IPython.display import Markdown, display
display(Markdown(open('outputs/atlas_run/report.md', encoding='utf-8').read()))

In [ ]:
# Run the unit + analysis test suite (perturbations, atlas, stats, attribution, emergence, ...).
!python -m pytest -q

## 2. Consistency clusters (the Reliability metric's input)

The Reliability metric is `accuracy x consistency` across semantically equivalent variants. Expand any benchmark into consistency clusters — every variant preserves the ground-truth answer (or, for numeric templates, recomputes it exactly).

In [ ]:
from instella_reasoning.perturbations import make_variants, PerturbationConfig
from instella_reasoning.records import read_benchmark

items = read_benchmark('examples/mini_benchmark.jsonl')
original = items[0]
print('ORIGINAL:', original.prompt, '\nanswer:', original.answer, '\n')
for v in make_variants(original, PerturbationConfig(seed=7)):
    print(f'[{v.variant_type}] {v.prompt}  (answer still {v.answer})')

## 3. Figures — the Reasoning Reliability Atlas

Install the `viz` extra and render the figure set (Atlas heatmap, accuracy-vs-consistency quadrant, contamination breakdown, attribution sources).

In [ ]:
!pip install -q -e ".[viz]"
!instella-reasoning plots \
  --scores outputs/atlas_run/scores.jsonl \
  --contamination outputs/atlas_run/contamination.jsonl \
  --output-dir outputs/atlas_run/figures

import glob
from IPython.display import Image, display
for path in sorted(glob.glob('outputs/atlas_run/figures/*.png')):
    print(path)
    display(Image(path))

## 4. FULL RUN on a T4 — real embeddings, real Instella, real data

Everything below needs a **GPU runtime** and downloads models/data from HuggingFace. The cells self-skip on CPU. See `docs/DATA_DOWNLOAD.md` for the complete AMD data procedure and `docs/COMPUTE.md` for the compute budget.

In [ ]:
# Detect GPU; install the heavy extras only when present.
import shutil, subprocess
HAS_GPU = shutil.which('nvidia-smi') is not None and subprocess.run(['nvidia-smi'], capture_output=True).returncode == 0
print('GPU available:', HAS_GPU)
if HAS_GPU:
    !pip install -q -e ".[hf,retrieval,viz]" bitsandbytes
else:
    print('No GPU -> skipping the full run. Sections 1-3 already demonstrate the whole pipeline on CPU.')

In [ ]:
# Download a benchmark and the Instella GSM8K-synthetic training shard (contamination ground truth).
if HAS_GPU:
    !instella-reasoning load-benchmark --benchmark gsm8k --output data/processed/gsm8k.jsonl --limit 200
    !instella-reasoning load-corpus --hf-path amd/Instella-GSM8K-synthetic --source instella-gsm8k-synthetic --output data/processed/gsm8k_synth.jsonl --limit 5000
    print('Downloaded GSM8K (200) and Instella-GSM8K-synthetic (5000).')

In [ ]:
# Embedding + 13-gram contamination scan with real MiniLM + FAISS.
if HAS_GPU:
    !instella-reasoning scan-contamination-embedding \
      --benchmark data/processed/gsm8k.jsonl \
      --corpus data/processed/gsm8k_synth.jsonl \
      --output outputs/gsm8k_contamination.jsonl --top-k 20

In [ ]:
# Generate with Instella-3B (4-bit), score, then build the accuracy gap.
if HAS_GPU:
    !instella-reasoning generate \
      --benchmark data/processed/gsm8k.jsonl \
      --model amd/Instella-3B \
      --output outputs/gsm8k_generations.jsonl --load-in-4bit
    !instella-reasoning score-generations \
      --benchmark data/processed/gsm8k.jsonl \
      --generations outputs/gsm8k_generations.jsonl \
      --output outputs/gsm8k_scores.jsonl
    !instella-reasoning accuracy-gap \
      --scores outputs/gsm8k_scores.jsonl \
      --contamination outputs/gsm8k_contamination.jsonl \
      --output outputs/gsm8k_accuracy_gap.json
    print(open('outputs/gsm8k_accuracy_gap.json', encoding='utf-8').read())

### Full config-driven run

For the real study, edit `configs/pipeline/full.yaml`: point `data.benchmark`/`data.corpus` at your downloaded JSONL, set `data.generations: null` and `generation.model: amd/Instella-3B`, set `generation.load_in_4bit: true`, and turn on `data.expand_variants: true` to build consistency clusters. Then a single command produces the whole Atlas:

```bash
instella-reasoning run-all --config configs/pipeline/full.yaml
```